# GSE295514 - RDS を読む（R kernel）

`GSE295514_ALS_mouse_brain.rds` を対話的に開き、class / assays / `meta.data` を確認してから、Python が読める中間ファイルを `data/intermediate_from_r/GSE295514/` に書き出す:

- `counts.mtx`（genes x cells, Matrix Market）
- `metadata.csv`
- `genes.csv`（gene_symbol）
- `barcodes.csv`（cell_id）

Python ノートブック 01 が `io_dense.read_from_r_intermediate(...)` でこれを読む。

必要: **R Jupyter kernel (IRkernel)** と `Matrix`、対象に応じて `Seurat`/`SeuratObject` または `SingleCellExperiment`。

In [2]:
library(Matrix)

# rds を読み込み、まずクラスを確認
rds_path <- "../../data/raw/GSE295514/GSE295514_ALS_mouse_brain.rds"
obj <- readRDS(rds_path)

class(obj)

[1] "Seurat"
attr(,"package")
[1] "SeuratObject"

In [2]:
names(obj)
tryCatch(slotNames(obj), error = function(e) e$message)

Loading required namespace: SeuratObject



[1] "RNA"

[1] "assays"       "meta.data"    "active.assay" "active.ident" "graphs"      
 [6] "neighbors"    "reductions"   "images"       "project.name" "misc"        
[11] "version"      "commands"     "tools"

## Seurat の場合

In [3]:
library(Seurat)
Assays(obj)            # assay 一覧
DefaultAssay(obj)
head(obj@meta.data)    # meta.data の列
colnames(obj@meta.data)

[1] "RNA"

[1] "RNA"

,orig.ident,nCount_RNA,nFeature_RNA
,<chr>,<dbl>,<int>
GEX_S1_ALS-FC1_AAACCCAAGAAACCCG-1,GEX_S1_ALS-FC1,12547,4470
GEX_S1_ALS-FC1_AAACCCAAGCGTGCTC-1,GEX_S1_ALS-FC1,4287,1805
GEX_S1_ALS-FC1_AAACCCAAGGAGTCTG-1,GEX_S1_ALS-FC1,3255,1598
GEX_S1_ALS-FC1_AAACCCAAGGTAAGGA-1,GEX_S1_ALS-FC1,4995,1617
GEX_S1_ALS-FC1_AAACCCAAGTGGCCTC-1,GEX_S1_ALS-FC1,9165,3142
GEX_S1_ALS-FC1_AAACCCACACAACGTT-1,GEX_S1_ALS-FC1,4096,1800


[1] "orig.ident"   "nCount_RNA"   "nFeature_RNA"

In [8]:
obj <- JoinLayers(obj, assay = DefaultAssay(obj), layers = "counts")
counts <- GetAssayData(obj, assay = DefaultAssay(obj), layer = "counts")
dim(counts); counts[1:5, 1:5]

[1]  24134 123401

5 x 5 sparse Matrix of class "dgCMatrix"
        GEX_S1_ALS-FC1_AAACCCAAGAAACCCG-1 GEX_S1_ALS-FC1_AAACCCAAGCGTGCTC-1
Xkr4                                    .                                 .
Gm19938                                 .                                 .
Rp1                                     1                                 .
Sox17                                   .                                 .
Gm37587                                 .                                 .
        GEX_S1_ALS-FC1_AAACCCAAGGAGTCTG-1 GEX_S1_ALS-FC1_AAACCCAAGGTAAGGA-1
Xkr4                                    .                                 .
Gm19938                                 .                                 .
Rp1                                     .                                 .
Sox17                                   .                                 .
Gm37587                                 .                                 .
        GEX_S1_ALS-FC1_AAACCCAAGTGGCCTC-1
Xkr4 

## SingleCellExperiment の場合

In [ ]:
# library(SingleCellExperiment)
# assayNames(obj)
# counts <- assay(obj, "counts")
# colData(obj); rowData(obj)

## Python 用の中間ファイルを書き出す

上のどちらかで `counts`(genes x cells) と `meta`(細胞メタ) を作ってから実行。

In [9]:
out_dir <- "../../data/intermediate_from_r/GSE295514"
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

# --- オブジェクトに応じて counts / meta を取り出す ---
if (inherits(obj, "Seurat")) {
    obj <- JoinLayers(obj, assay = DefaultAssay(obj), layers = "counts")
    counts <- GetAssayData(obj, assay = DefaultAssay(obj), layer = "counts")
    meta <- obj@meta.data
} else if (inherits(obj, "SingleCellExperiment")) {
    counts <- SummarizedExperiment::assay(obj, "counts")
    meta <- as.data.frame(SummarizedExperiment::colData(obj))
} else {
    stop(paste("未対応のクラス:", paste(class(obj), collapse = ", ")))
}

counts <- as(counts, "CsparseMatrix")
writeMM(counts, file.path(out_dir, "counts.mtx"))
write.csv(meta, file.path(out_dir, "metadata.csv"))
write.csv(data.frame(gene_symbol = rownames(counts)),
          file.path(out_dir, "genes.csv"), row.names = FALSE)
write.csv(data.frame(cell_id = colnames(counts)),
          file.path(out_dir, "barcodes.csv"), row.names = FALSE)

cat("中間ファイルを書き出しました:", out_dir, "\n")
dim(counts)

NULL

中間ファイルを書き出しました: ../../data/intermediate_from_r/GSE295514 


[1]  24134 123401

この後 **notebooks/python/01** の GSE295514 セル （`io_dense.read_from_r_intermediate(...)`）で読み込む。